# Type II Supernova End-to-End Simulation

Simulates the hydrogen-bearing (Type II) core-collapse supernova populations together, as one catch-all sample:

| Population | Class | SED | Rate | Redshift limit | Duration |
|---|---|---|---|---|---|
| Type IIP | `TypeIIPSNe` | `TypeIIPSED` (two-exponential + radioactive-tail light curve x cooling blackbody) | 48.7% of the core-collapse rate (69.6% Type II x 69.9% IIP; Shivvers et al. 2017, Li et al. 2011) | z=0.8 | 100 d |
| Type IIP + early excess | `TypeIIPExcessSNe` | `TypeIIPExcessSED` | 30% of the IIP rate (14.6% of the core-collapse rate; Bruch et al. 2023, ZTF) | z=1.2 | 100 d |
| Type IIb | `TypeIIbSNe` | `TypeIIbSED` (two Bazin pulses x cooling blackbody) | 10.3% of the core-collapse rate (Shivvers et al. 2017) | z=0.5 | 200 d |

This replaces the separate Type IIP and Type IIP-excess notebooks. This notebook is fully
self-contained: it downloads the default UVEX schedule, samples a Monte Carlo population of all three
against it, screens the population down to what UVEX would actually detect, and plots the result --
broken down by population -- using the same pipeline the `uvex-transients` CLI runs from a config file
(see `configs/quickstart_tde.yaml`, `configs/full_run.yaml`), just driven from Python so you can poke at
every intermediate object.

Run all cells top to bottom.

## Setup

Load the default schedule and configure the transient populations. Every population is registered with the same `SurveySimulator`, so one call samples and screens all of them against the schedule together; each event keeps a `transient_type` tag saying which population it came from.

In [ ]:
import numpy as np
from astropy import units as u
from matplotlib import pyplot as plt

# `uvex_transients` installs warning filters for some noisy-but-harmless dependency
# warnings (e.g. lal's Jupyter/IPython SWIG redirect notice) as soon as it's imported
# -- import it before `m4opt`/`ligo.skymap` (which trigger that warning) so the filter
# is already in place.
from uvex_transients.simulation.core import SurveySimulator
from uvex_transients.surveys import get_schedule
from uvex_transients.utils.plotting import add_funnel_legend, compute_funnel_bounds, plot_detection_funnel
from uvex_transients.transients.supernovae import TypeIIbSNe, TypeIIPExcessSNe, TypeIIPSNe

from m4opt.missions import uvex

schedule = get_schedule()
transients = {
    "iip_sne": TypeIIPSNe(),
    "iip_excess_sne": TypeIIPExcessSNe(),
    "iib_sne": TypeIIbSNe(),
}
simulator = SurveySimulator(schedule, transients=transients, simulation_seed=42)

LABELS = {"iip_sne": "Type IIP", "iip_excess_sne": "Type IIP + excess", "iib_sne": "Type IIb"}
COLORS = {"iip_sne": "#4C72B0", "iip_excess_sne": "#C44E52", "iib_sne": "#55A868"}

for key, transient in transients.items():
    print(
        f"{key:16s} redshift_limit={transient.redshift_limit}, duration_limit={transient.duration_limit}, sed={type(transient.sed).__name__}"
    )

## Sample a Monte Carlo population

`generate_events` uses **windowed sampling**: it only draws events within HEALPix pixels/time
bins the schedule could plausibly have caught, given each transient's own duration window.

In [ ]:
# Type IIP SNe are the highest-rate population here, so downsample aggressively to keep the freshly-sampled catalog tractable.
# One factor applies to every population at once; the counts below are scaled back up by it.
TIME_BINS = 200
NSIDE = 256
DOWNSAMPLE = 200
SCALE = DOWNSAMPLE or 1  # every count below is multiplied back up by the downsampling factor

catalog = simulator.generate_events(time_bins=TIME_BINS, nside=NSIDE, downsample=DOWNSAMPLE)
print(f"Sampled {len(catalog) * SCALE:,} events across {TIME_BINS} time bin(s) at NSIDE={NSIDE}.")
for key in transients:
    print(f"  {key:16s} {np.sum(catalog.transient_type == key) * SCALE:>10,}")

## Screen the population

Two progressively more expensive passes narrow the freshly-sampled catalog down to what matters:
first, a cheap check of whether an event could *ever* clear a fixed magnitude limit
(`filter_by_limiting_magnitude`); then the real question of whether it was actually detected
above a given SNR by an observation the schedule made (`filter_by_snr`).

In [ ]:
MAG_LIMIT = 25.0
SNR_THRESHOLD = 5.0

mag_filtered = simulator.filter_by_limiting_magnitude(catalog, uvex, mag_limit=MAG_LIMIT)
print(f"{len(mag_filtered) * SCALE:,} could ever clear {MAG_LIMIT} AB mag.")

detected = simulator.filter_by_snr(mag_filtered, uvex, snr_threshold=SNR_THRESHOLD)
print(f"{len(detected) * SCALE:,} were detected above SNR={SNR_THRESHOLD}.")

## Detection funnel

How many events of each population survive each stage, and what fraction of the sampled events UVEX detects.

In [ ]:
stage_names = ["Sampled", f"Mag < {MAG_LIMIT}", f"SNR > {SNR_THRESHOLD}"]
stages = [catalog, mag_filtered, detected]
raw_counts = {key: [int(np.sum(stage.transient_type == key)) for stage in stages] for key in transients}

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.8 / len(transients)
for i, key in enumerate(transients):
    scale = SCALE
    counts_i = [c * scale for c in raw_counts[key]]
    mc_lower, mc_upper, rate_lower, rate_upper = compute_funnel_bounds(raw_counts[key], rate_ci=transients[key].RATE_CI)
    x = np.arrange(len(stage_names)) + (i - (len(transients) - 1) / 2) * width
    plot_detection_funnel(
        ax,
        x=x,
        counts=np.asarray(counts_i, dtype=float),
        mc_lower=mc_lower * scale,
        mc_upper=mc_upper * scale,
        rate_lower=rate_lower * scale,
        rate_upper=rate_upper * scale,
        color=COLORS[key],
        width=width,
        label=LABELS[key],
    )
    for xi, count in zip(x, counts_i):
        ax.text(xi, count, f"{count:,}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(np.arrange(len(stage_names)), stage_names)
ax.set_yscale("log")
ax.set_ylabel("Number of events")
ax.set_title("Detection funnel")
add_funnel_legend(ax)
fig.tight_layout()

print(f"{'population':<28s}{'sampled':>12s}{'detected':>12s}{'fraction':>12s}")
for key in transients:
    scale = SCALE
    sampled, det = raw_counts[key][0] * scale, raw_counts[key][-1] * scale
    print(f"{LABELS[key]:<28s}{sampled:>12,}{det:>12,}{det / max(sampled, 1):>12.2%}")

Black error bars are the **MC (statistical)** uncertainty on each stage's count -- treating it as a binomial subsample of the raw simulated draws (Clopper-Pearson). The pale shaded band is the **rate (systematic)** uncertainty from each population's own literature rate normalization (`RATE_CI`), which scales every stage by the same factor rather than shrinking as the sample is cut down.


## Yield summary

Tabulate each population's footprint-aware exposure (`SurveySimulator.compute_effective_exposure`,
an `ExposureCatalog`), then combine it with `catalog`/`detected` into a per-population yield
estimate (`EventCatalog.compute_yield_summary`, a `YieldTable`) -- rate, intrinsic UVEX event
count, detection probability, and expected detections, each with both Clopper-Pearson (Monte
Carlo) and rate-normalization confidence bounds. This is the same summary the `uvex-transients run`
CLI command writes to `exposure.ecsv`/`yield_summary.ecsv`/`yield_summary.txt`.
`YieldTable.display_expected_detections` renders just each population's expected-detections
estimate, with both uncertainties stacked, as typeset LaTeX.

In [ ]:
exposure = simulator.compute_effective_exposure(time_bins=TIME_BINS, nside=NSIDE)
yield_table = catalog.compute_yield_summary(detected, exposure, transients)
yield_table.display_expected_detections()

## Sky distribution

In [ ]:
fig = plt.figure(figsize=(8, 4))
ax = fig.add_subplot(111, projection="aitoff")
ax.grid(True)

ra_sampled = catalog.coord.ra.wrap_at(180 * u.deg).radian
ax.scatter(
    ra_sampled, catalog.coord.dec.radian, s=2, alpha=0.2, color="#888888", label=f"Sampled ({SCALE * len(catalog):,})"
)
for key in transients:
    sel = detected.transient_type == key
    ax.scatter(
        detected.coord.ra.wrap_at(180 * u.deg).radian[sel],
        detected.coord.dec.radian[sel],
        s=8,
        color=COLORS[key],
        label=f"{LABELS[key]} detected ({SCALE * int(sel.sum()):,})",
    )
ax.legend(loc="lower right", markerscale=2, fontsize=7)
ax.set_title("Sky distribution")
fig.tight_layout()

## Redshift distribution

Each population is sampled out to its own redshift limit; the detected events fall well inside it.

In [ ]:
fig, axes = plt.subplots(1, len(transients), figsize=(4 * len(transients), 3.6), sharey=True)
for ax, (key, transient) in zip(np.atleast_1d(axes), transients.items()):
    bins = np.linspace(0, transient.redshift_limit, 30)
    sampled_z = catalog.redshift[catalog.transient_type == key]
    detected_z = detected.redshift[detected.transient_type == key]
    ax.hist(sampled_z, bins=bins, color="#888888", label=f"Sampled ({len(sampled_z)})")
    ax.hist(detected_z, bins=bins, color=COLORS[key], label=f"Detected ({len(detected_z)})")
    ax.set_yscale("log")
    ax.set_xlabel("Redshift")
    ax.set_title(LABELS[key])
    ax.legend(fontsize=8)
np.atleast_1d(axes)[0].set_ylabel("Number of events (before rescaling)")
fig.tight_layout()

## Example light curves

Reconstruct one event of each population as a real `Event` (`EventCatalog.get_events`) and run full
synthetic photometry (`Event.simulate_photometry`) against every observation the schedule actually made of
it. Prefer a detected event, but each population's detected count depends on the specific random
draw above -- fall back to the best available stage (mag-screened, then the raw sample) if
none was detected. Filled squares are detections; open triangles are upper limits; the lines are the
noiseless model.

In [ ]:
def plot_example(key, seed=1):
    """Plot one reconstructed event of population `key`, drawn from the deepest stage that has one."""
    rng = np.random.default_rng(seed)
    for label, source in (("detected", detected), ("mag-screened", mag_filtered), ("sampled", catalog)):
        ids = source.event_id[source.transient_type == key]
        if len(ids) > 0:
            stage = label
            break
    else:
        raise RuntimeError(
            f"No {key} events at all were sampled -- try a larger TIME_BINS/NSIDE or a smaller DOWNSAMPLE."
        )

    event = source.get_events(int(rng.choice(ids)), transients, schedule)
    print(f"Example {LABELS[key]} event (from the {stage!r} stage):")
    print(event)

    phot = event.simulate_photometry(uvex)
    t_since_explosion = (phot["obs_time"] - event.t_explosion).to(u.day)
    t_theory = np.linspace(0, transients[key].duration_limit.to_value(u.day), 300) * u.day

    fig, ax = plt.subplots(figsize=(7, 4))
    for band, color in {"FUV": "#4C72B0", "NUV": "#DD8452"}.items():
        ax.plot(t_theory.value, event.mag(t_theory, uvex, band=band).value, color=color, lw=1.5, alpha=0.6)

        in_band = np.isfinite(phot["ab_mag"]) & (phot["band"] == band)
        detected_pts = in_band & (phot["snr"] > SNR_THRESHOLD)
        upper_limits = in_band & (phot["snr"] <= SNR_THRESHOLD)

        if np.any(detected_pts):
            ax.errorbar(
                t_since_explosion[detected_pts].value,
                phot["ab_mag"][detected_pts],
                yerr=5 * phot["mag_err"][detected_pts],
                marker="s",
                mfc=color,
                mec="k",
                ecolor=color,
                linestyle="none",
                label=band,
            )
        if np.any(upper_limits):
            ax.errorbar(
                t_since_explosion[upper_limits].value,
                phot["ab_mag"][upper_limits],
                yerr=[
                    phot["mag_upper"][upper_limits] - phot["ab_mag"][upper_limits],
                    np.abs(phot["mag_lower"][upper_limits] - phot["ab_mag"][upper_limits]),
                ],
                marker="v",
                mfc="w",
                mec=color,
                ecolor=color,
                linestyle="none",
            )

    ax.invert_yaxis()
    ax.set_xlabel("Days since explosion")
    ax.set_ylabel("AB magnitude")
    ax.set_title(f"{LABELS[key]}: event {event.event_id} (z={event.redshift:.3f}, {event.n_observations} observations)")
    ax.legend()
    fig.tight_layout()
    plt.show()

### Type IIP

In [ ]:
plot_example("iip_sne")

### Type IIP + excess

In [ ]:
plot_example("iip_excess_sne")

### Type IIb

In [ ]:
plot_example("iib_sne")